# Agente Experto en analizar económicamente el sector de la industria textil y la moda en España
**Stack:** Google Gemini · ChromaDB · LangGraph · RAG + Memoria  
**Dominio:** Industria textil y moda en España (5 documentos PDF)

---
## PASO 1 — Carga y preprocesamiento de documentos PDF

#### Imports y configuración inicial

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print(f"API key cargada")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

DOCS_DIR = Path("..") / "docs"

pdf_files = [
    DOCS_DIR / "informe-economico-de-la-moda-en-espana-2025.pdf",
    DOCS_DIR / "presentaciones_sectoriales_textil.pdf",
    DOCS_DIR / "memoria_anual_inditex_2025.pdf",
    DOCS_DIR / "circularidad_textil.pdf",
    DOCS_DIR / "comercio_textil_2024.pdf",
]

documents = []
for path in pdf_files:
    if not path.exists():
        print(f"⚠ No encontrado: {path}")
        continue
    loader = PyPDFLoader(str(path))
    docs = loader.load()
    documents.extend(docs)
    print(f"✓ {path.name:<55} {len(docs):>3} páginas")

print(f"\nTotal páginas cargadas: {len(documents)}")

In [ ]:
# Inspección de un documento
sample = documents[0]
print("=== Metadatos ===")
print(sample.metadata)
print("\n=== Primeros 5 caracteres ===")
print(sample.page_content[:5])

##### Chunking con RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len,
)

chunks = splitter.split_documents(documents)

print(f"Total chunks generados:    {len(chunks)}")
print(f"Promedio chars/chunk:      {sum(len(c.page_content) for c in chunks) // len(chunks)}")

print("\n=== Ejemplo chunk #20 ===")
print(f"Fuente: {Path(chunks[20].metadata.get('source', 'N/A')).name} | Página: {chunks[20].metadata.get('page', 'N/A')}")
print(chunks[20].page_content)

##### Crear embeddings con Gemini e indexar en ChromaDB

In [ ]:
import shutil
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

EMBEDDING_MODEL = "models/text-embedding-004"
CHROMA_DIR = str(Path("..") / "chroma_db")
COLLECTION_NAME = "textil_moda_espana"

# Borrar colección persistida para evitar duplicados al re-ejecutar
if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)
    print(f"✓ Colección anterior eliminada ({CHROMA_DIR})")

print(f"Cargando embeddings Gemini: {EMBEDDING_MODEL}\n")

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GEMINI_API_KEY,
)

print(f"Indexando {len(chunks)} chunks en ChromaDB...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

total = vectorstore._collection.count()
print(f"✓ Colección '{COLLECTION_NAME}' creada con Gemini Embeddings ({EMBEDDING_MODEL})")
print(f"✓ Chunks indexados: {total}")
print(f"✓ Persistido en:    {CHROMA_DIR}")

##### Verificación (consultas de prueba)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.7},
)

test_queries = [
    "¿Cuál es el volumen del sector textil en España?",
    "¿Qué estrategia de sostenibilidad tiene Inditex?",
    "¿Qué es la circularidad en el sector textil?",
]

for query in test_queries:
    results = retriever.invoke(query)
    print(f"\n{'='*65}")
    print(f"QUERY: {query}")
    print(f"{'='*65}")
    for i, doc in enumerate(results):
        fuente = Path(doc.metadata.get('source', 'N/A')).name
        pagina = doc.metadata.get('page', 'N/A')
        print(f"  [{i+1}] {fuente} | p.{pagina}")
        print(f"      {doc.page_content[:200].strip()}")
        print()

print("✓ Base de conocimiento lista para conectar el agente")

---
## PASO 2 — Agente LangGraph con RAG y Memoria

Arquitectura del grafo:
```
START → [nodo_reformular] → [nodo_recuperar] → [nodo_generar] → END
```
- **nodo_reformular**: usa el LLM para reescribir la pregunta con sinónimos del dominio textil antes de buscar
- **nodo_recuperar**: busca los 6 chunks más relevantes en ChromaDB usando MMR (diversidad maximizada)
- **nodo_generar**: llama a Gemini con el system prompt + contexto recuperado + historial completo
- **Memoria**: `MemorySaver` persiste el historial entre turnos usando `thread_id`

### System Prompt
El prompt define el rol, tono y limitaciones del agente. Se inyecta en cada turno junto con el contexto RAG recuperado.

In [ ]:
# Celda 7 — System prompt del agente
SYSTEM_PROMPT = """Eres un analista experto en la industria textil y moda en España.

Tu conocimiento proviene de los documentos que se te proporcionan como contexto en cada turno.
Respondes siempre en español, con un tono profesional y conciso.

Reglas:
- Basa tus respuestas en el contexto documental proporcionado.
- Puedes sintetizar y relacionar información de tus respuestas anteriores en la conversación,
  siempre que las afirmaciones originales provengan del contexto documental.
- Si el contexto no contiene información suficiente y no puedes inferirlo del historial, indícalo claramente.
- No inventes datos, cifras ni estadísticas que no aparezcan en el contexto o en el historial previo.
- Cuando cites datos numéricos, menciona la fuente si está disponible en el contexto.
"""

print("System prompt definido.")

In [ ]:
# Celda 8 — LLM con Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

LLM_MODEL = "gemini-2.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    temperature=0.3,
    google_api_key=GEMINI_API_KEY,
)

print(f"LLM cargado: {LLM_MODEL}")

In [ ]:
# Celda 9 — Estado y nodos del grafo LangGraph
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph.message import add_messages

REFORMULATION_PROMPT = """Eres un asistente especializado en documentos sobre la industria textil y moda en España.
Tu tarea es reformular la siguiente pregunta expandiéndola con sinónimos y términos relacionados del sector, \
para mejorar la recuperación de información en una base de conocimiento documental.
Devuelve ÚNICAMENTE la query reformulada en una sola línea, sin explicaciones ni puntuación extra."""

class EstadoAgente(TypedDict):
    mensajes: Annotated[list[BaseMessage], add_messages]
    contexto: str
    query_reformulada: str

def nodo_reformular(estado: EstadoAgente) -> dict:
    """Reescribe la pregunta del usuario con sinónimos del dominio textil para mejorar el RAG."""
    ultima_pregunta = estado["mensajes"][-1].content
    respuesta = llm.invoke([
        SystemMessage(content=REFORMULATION_PROMPT),
        HumanMessage(content=ultima_pregunta),
    ])
    query_expandida = respuesta.content.strip()
    print(f"  [QUERY] Original:    '{ultima_pregunta[:70]}'")
    print(f"  [QUERY] Reformulada: '{query_expandida[:70]}'")
    return {"query_reformulada": query_expandida}

def nodo_recuperar(estado: EstadoAgente) -> dict:
    """Recupera los chunks más relevantes de ChromaDB usando la query reformulada."""
    docs = retriever.invoke(estado["query_reformulada"])
    contexto = "\n\n".join([
        f"[Fuente: {Path(d.metadata.get('source','?')).name} | p.{d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    ])
    print(f"  [RAG]   {len(docs)} chunks recuperados")
    return {"contexto": contexto}

def nodo_generar(estado: EstadoAgente) -> dict:
    """Genera la respuesta usando el LLM con el contexto RAG y el historial."""
    prompt_con_contexto = SYSTEM_PROMPT + f"\n\nCONTEXTO RELEVANTE DE LOS DOCUMENTOS:\n{estado['contexto']}"
    mensajes_completos = [SystemMessage(content=prompt_con_contexto)] + estado["mensajes"]
    respuesta = llm.invoke(mensajes_completos)
    return {"mensajes": [respuesta]}

print("Nodos definidos: nodo_reformular, nodo_recuperar, nodo_generar")

In [ ]:
# Celda 10 — Construir y compilar el grafo con memoria
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

grafo = StateGraph(EstadoAgente)

grafo.add_node("reformular", nodo_reformular)
grafo.add_node("recuperar", nodo_recuperar)
grafo.add_node("generar", nodo_generar)

grafo.add_edge(START, "reformular")
grafo.add_edge("reformular", "recuperar")
grafo.add_edge("recuperar", "generar")
grafo.add_edge("generar", END)

memoria = MemorySaver()
agente = grafo.compile(checkpointer=memoria)

# Visualización del grafo
try:
    from IPython.display import Image, display
    display(Image(agente.get_graph().draw_mermaid_png()))
except Exception:
    print(agente.get_graph().draw_ascii())

print("\n✓ Agente compilado con memoria")

---
## PASO 3 — Interacción y ejemplos documentados

In [ ]:
# Celda 11 — Función de conversación con el agente
def chatear(pregunta: str, thread_id: str = "sesion-01") -> str:
    """Envía una pregunta al agente y devuelve la respuesta."""
    config = {"configurable": {"thread_id": thread_id}}
    resultado = agente.invoke(
        {"mensajes": [HumanMessage(content=pregunta)]},
        config=config,
    )
    return resultado["mensajes"][-1].content

def mostrar_respuesta(pregunta: str, thread_id: str = "sesion-01"):
    """Helper para mostrar pregunta y respuesta con formato."""
    print(f"\n{'='*65}")
    print(f"PREGUNTA: {pregunta}")
    print(f"{'='*65}")
    respuesta = chatear(pregunta, thread_id)
    print(f"\nRESPUESTA:\n{respuesta}")
    return respuesta

print("✓ Funciones de conversación listas")

In [12]:
# Celda 12 — 5 ejemplos documentados (mismo thread para demostrar memoria)
THREAD = "demo-ejemplos"

# Ejemplo 1
mostrar_respuesta("¿Cuántas empresas textiles hay en España y cómo ha evolucionado este número?", THREAD)


PREGUNTA: ¿Cuántas empresas textiles hay en España y cómo ha evolucionado este número?
  [RAG] 6 chunks recuperados para: '¿Cuántas empresas textiles hay en España y cómo ha evolucion...'

RESPUESTA:
Según los datos disponibles, en 2024 hay 7.547 empresas en el sector Textil/Confección en España.

La evolución del número de empresas en este sector ha sido la siguiente:
*   2017: 7.823 empresas
*   2018: 8.337 empresas
*   2019: 8.282 empresas
*   2020: 8.119 empresas
*   2021: 7.957 empresas
*   2022: 7.823 empresas
*   2023: 7.600 empresas
*   2024: 7.547 empresas

(Fuente: comercio_textil_2024.pdf | p.47)


'Según los datos disponibles, en 2024 hay 7.547 empresas en el sector Textil/Confección en España.\n\nLa evolución del número de empresas en este sector ha sido la siguiente:\n*   2017: 7.823 empresas\n*   2018: 8.337 empresas\n*   2019: 8.282 empresas\n*   2020: 8.119 empresas\n*   2021: 7.957 empresas\n*   2022: 7.823 empresas\n*   2023: 7.600 empresas\n*   2024: 7.547 empresas\n\n(Fuente: comercio_textil_2024.pdf | p.47)'

In [13]:
# Ejemplo 2
mostrar_respuesta("¿Cuál es la estrategia medioambiental de Inditex?", THREAD)


PREGUNTA: ¿Cuál es la estrategia medioambiental de Inditex?
  [RAG] 6 chunks recuperados para: '¿Cuál es la estrategia medioambiental de Inditex?...'

RESPUESTA:
La estrategia medioambiental de Inditex se fundamenta en su Política de Sostenibilidad, la cual integra principios medioambientales y sociales. La compañía ha desarrollado un modelo de negocio flexible e integrado, con una fuerte orientación al cliente y un claro enfoque en la sostenibilidad.

Dentro de esta estrategia, Inditex realiza actividades de investigación, desarrollo e innovación (I+D+I) para mejorar procesos de fabricación y distribución, así como para desarrollar tecnologías que faciliten la gestión de sus negocios. Un ejemplo concreto de sus objetivos es la reducción de un 25% del consumo de agua en la cadena de suministro para el año 2025.

Adicionalmente, Inditex se enfoca en el ecodiseño, considerando el medioambiente como un factor clave desde la fase de diseño del producto y a lo largo de todas las etapas de 

'La estrategia medioambiental de Inditex se fundamenta en su Política de Sostenibilidad, la cual integra principios medioambientales y sociales. La compañía ha desarrollado un modelo de negocio flexible e integrado, con una fuerte orientación al cliente y un claro enfoque en la sostenibilidad.\n\nDentro de esta estrategia, Inditex realiza actividades de investigación, desarrollo e innovación (I+D+I) para mejorar procesos de fabricación y distribución, así como para desarrollar tecnologías que faciliten la gestión de sus negocios. Un ejemplo concreto de sus objetivos es la reducción de un 25% del consumo de agua en la cadena de suministro para el año 2025.\n\nAdicionalmente, Inditex se enfoca en el ecodiseño, considerando el medioambiente como un factor clave desde la fase de diseño del producto y a lo largo de todas las etapas de su ciclo de vida.\n\n(Fuentes: memoria_anual_inditex_2025.pdf | p.64, p.79; circularidad_textil.pdf | p.47)'

In [14]:
# Ejemplo 3
mostrar_respuesta("¿Qué es la economía circular aplicada al sector textil?", THREAD)


PREGUNTA: ¿Qué es la economía circular aplicada al sector textil?
  [RAG] 6 chunks recuperados para: '¿Qué es la economía circular aplicada al sector textil?...'

RESPUESTA:
La economía circular aplicada al sector textil se centra en la transición hacia un modelo que minimiza la generación de residuos y maximiza la vida útil de los productos. Esto implica un cambio desde el modelo lineal de "usar y tirar" hacia un ciclo donde los materiales se reutilizan y reciclan.

Los pilares de esta aplicación incluyen:

*   **Reducción de residuos:** Se busca disminuir la cantidad de desechos textiles generados.
*   **Creación de nuevos productos:** Los materiales de desecho se transforman en nuevos productos industriales.
*   **Nuevos modelos de negocio:** Surgen oportunidades como el alquiler de prendas o la reventa de productos de segunda mano.
*   **Extensión de la vida útil:** Se promueven iniciativas para alargar la durabilidad de las prendas.

Este enfoque es impulsado por la Comisión Euro

'La economía circular aplicada al sector textil se centra en la transición hacia un modelo que minimiza la generación de residuos y maximiza la vida útil de los productos. Esto implica un cambio desde el modelo lineal de "usar y tirar" hacia un ciclo donde los materiales se reutilizan y reciclan.\n\nLos pilares de esta aplicación incluyen:\n\n*   **Reducción de residuos:** Se busca disminuir la cantidad de desechos textiles generados.\n*   **Creación de nuevos productos:** Los materiales de desecho se transforman en nuevos productos industriales.\n*   **Nuevos modelos de negocio:** Surgen oportunidades como el alquiler de prendas o la reventa de productos de segunda mano.\n*   **Extensión de la vida útil:** Se promueven iniciativas para alargar la durabilidad de las prendas.\n\nEste enfoque es impulsado por la Comisión Europea a través del Circular Economy Action Plan y la Estrategia Europea para los textiles sostenibles.\n\n(Fuentes: circularidad_textil.pdf | p.19, p.5)'

In [15]:
# Ejemplo 4 — referencia al contexto de preguntas anteriores (demuestra memoria)
mostrar_respuesta("¿Y cómo se compara esa estrategia de Inditex con los principios de economía circular que acabas de explicar?", THREAD)


PREGUNTA: ¿Y cómo se compara esa estrategia de Inditex con los principios de economía circular que acabas de explicar?
  [RAG] 6 chunks recuperados para: '¿Y cómo se compara esa estrategia de Inditex con los princip...'

RESPUESTA:
La estrategia medioambiental de Inditex se alinea con los principios de la economía circular en varios aspectos. Si bien el contexto proporcionado no detalla explícitamente todas las acciones de Inditex en relación con cada principio de la economía circular, sí se pueden inferir conexiones:

*   **Enfoque en la sostenibilidad:** La Política de Sostenibilidad de Inditex, que incluye principios medioambientales, es la base de su estrategia. Esto concuerda con el objetivo general de la economía circular de operar de manera más sostenible.
*   **I+D+I para mejorar procesos:** La inversión en investigación, desarrollo e innovación para mejorar procesos de fabricación y distribución puede contribuir a la eficiencia de recursos y a la reducción de residuos, aspect

'La estrategia medioambiental de Inditex se alinea con los principios de la economía circular en varios aspectos. Si bien el contexto proporcionado no detalla explícitamente todas las acciones de Inditex en relación con cada principio de la economía circular, sí se pueden inferir conexiones:\n\n*   **Enfoque en la sostenibilidad:** La Política de Sostenibilidad de Inditex, que incluye principios medioambientales, es la base de su estrategia. Esto concuerda con el objetivo general de la economía circular de operar de manera más sostenible.\n*   **I+D+I para mejorar procesos:** La inversión en investigación, desarrollo e innovación para mejorar procesos de fabricación y distribución puede contribuir a la eficiencia de recursos y a la reducción de residuos, aspectos clave de la circularidad.\n*   **Reducción del consumo de agua:** El objetivo de reducir el consumo de agua en la cadena de suministro (25% para 2025) es una medida concreta que aborda la gestión eficiente de recursos, un pila

In [16]:
# Ejemplo 5
mostrar_respuesta("¿Cuál es la balanza comercial del sector textil en España y qué tendencia muestra?", THREAD)


PREGUNTA: ¿Cuál es la balanza comercial del sector textil en España y qué tendencia muestra?
  [RAG] 6 chunks recuperados para: '¿Cuál es la balanza comercial del sector textil en España y ...'

RESPUESTA:
El contexto proporcionado no contiene información específica sobre la balanza comercial del sector textil en España ni su tendencia.

(Fuentes: No disponibles en el contexto proporcionado.)


'El contexto proporcionado no contiene información específica sobre la balanza comercial del sector textil en España ni su tendencia.\n\n(Fuentes: No disponibles en el contexto proporcionado.)'

---
## PASO 4 — Chat interactivo

Celda de conversación libre con el agente. Ejecuta y escribe tu pregunta cuando aparezca el prompt.

In [17]:
# Celda interactiva — escribe 'salir' para terminar
THREAD_CHAT = "chat-interactivo"
print("Agente Textil España — escribe 'salir' para terminar\n")

while True:
    pregunta = input("Tú: ").strip()
    if not pregunta:
        continue
    if pregunta.lower() in ("salir", "exit", "quit"):
        print("Conversación finalizada.")
        break
    respuesta = chatear(pregunta, THREAD_CHAT)
    print(f"\nAgente: {respuesta}\n")

Agente Textil España — escribe 'salir' para terminar

Conversación finalizada.
